# Sapan Veri Analizi Ve Tahmin Modeli

Burada "stalk-root" adlı sütunda eksik veriler var bu eksik verileri doldurmak için bir makine öğrenimi modeli kullanarak bir çözüm geliştiriyoruz.
Eksik verileri "?" işareti ile belirledik ve bunları NaN (NumPy'nin temsil ettiği eksik değerler) olarak değiştirdik.
"stalk-root" sütununda eksik ve mevcut verileri ayırarak, eksik verilere sahip olanları bir DataFrame'de sakladık.
Modelimizi eğitmek için eksik olmayan verileri kullanarak özellikler ve hedef değişkeni ayırdık.
RandomForestClassifier modelini kullanarak bir makine öğrenimi modeli oluşturduk ve eğittik.
Oluşturduğumuz modeli kullanarak eksik değerleri tahmin ettik
Modelin doğruluğunu test ederek performansını değerlendirdik

In [19]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder

# Veri setini yükleme
df = pd.read_csv('mushrooms.csv')

# '?' işaretlerini NaN olarak işaretleyelim
df['stalk-root'] = df['stalk-root'].replace('?', np.nan)

# Eksik verilere sahip ve olmayan örnekleri ayırma
missing_data = df[df['stalk-root'].isnull()]
available_data = df.dropna(subset=['stalk-root'])

# One-Hot Encoding işlemi
X = pd.get_dummies(available_data.drop('stalk-root', axis=1))
y = available_data['stalk-root']

# Eğitim ve test verisi olarak ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Rastgele orman modeli oluşturma
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Tahmin yapma
y_pred = model.predict(X_test)

# Modelin doğruluğunu değerlendirme
accuracy = accuracy_score(y_test, y_pred)
print("Model Doğruluğu:", accuracy)


Model Doğruluğu: 1.0


In [20]:
# Eksik verilere sahip olan veri setindeki kategorik değişkenleri One-Hot Encoding işlemi ile dönüştürme
missing_data_encoded = pd.get_dummies(missing_data.drop('stalk-root', axis=1))

# Eğitim verisindeki sütun sayısına göre eksik verilere sahip olan veri setini yeniden şekillendirme
missing_data_encoded = missing_data_encoded.reindex(columns=X.columns, fill_value=0)

# Eksik değerleri doldurma
predicted_values = model.predict(missing_data_encoded)
missing_data['stalk-root'] = predicted_values

# Eksik verilere sahip olan veri setini tamamlama
completed_data = pd.concat([available_data, missing_data])

# Tamamlanmış veri setini kontrol etme
print("Tamamlanmış Veri Seti:")
print(completed_data.head())


Tamamlanmış Veri Seti:
  class cap-shape cap-surface cap-color bruises odor gill-attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b           s         w       t    l               f   
3     p         x           y         w       t    p               f   
4     e         x           s         g       f    n               f   

  gill-spacing gill-size gill-color  ... stalk-surface-below-ring  \
0            c         n          k  ...                        s   
1            c         b          k  ...                        s   
2            c         b          n  ...                        s   
3            c         n          n  ...                        s   
4            w         b          k  ...                        s   

  stalk-color-above-ring stalk-color-below-ring veil-type veil-color  \
0                      w                      w         p

C:\Users\Burhan Yasa\AppData\Local\Temp\ipykernel_35368\330971601.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_data['stalk-root'] = predicted_values


Veriyi kontrol etme aşaması

In [21]:
print("Tamamlanmış Veri Seti:")
print(completed_data)


Tamamlanmış Veri Seti:
     class cap-shape cap-surface cap-color bruises odor gill-attachment  \
0        p         x           s         n       t    p               f   
1        e         x           s         y       t    a               f   
2        e         b           s         w       t    l               f   
3        p         x           y         w       t    p               f   
4        e         x           s         g       f    n               f   
...    ...       ...         ...       ...     ...  ...             ...   
8119     e         k           s         n       f    n               a   
8120     e         x           s         n       f    n               a   
8121     e         f           s         n       f    n               a   
8122     p         k           y         n       f    y               f   
8123     e         x           s         n       f    n               a   

     gill-spacing gill-size gill-color  ... stalk-surface-below-ring  \
0   

In [22]:
print("stalk-root Sütunu:")
print(completed_data['stalk-root'])


stalk-root Sütunu:
0       e
1       c
2       c
3       e
4       e
       ..
8119    b
8120    b
8121    b
8122    b
8123    b
Name: stalk-root, Length: 8124, dtype: object


Sonuç olarak ? değerlerinin dağılımı ve stalk root sutunun son hali

In [23]:
# stalk-root sütunundaki değerlerin sayısını görmek
print(completed_data['stalk-root'].value_counts())


b    6224
e    1152
c     556
r     192
Name: stalk-root, dtype: int64
